# 言語をまたいで検索する (クロスリンガル検索)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_search/cross_lingual_search.ipynb)

NLP クックブック 18-13 ／ 検索

日本語のクエリで英語の文書を、あるいは英語のクエリで日本語の文書を検索したい。翻訳せずに、多言語の埋め込みで直接探したい。

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
docs = [
    ("en", "The Transformer architecture relies entirely on self-attention and dispenses with recurrence."),
    ("en", "Mount Fuji is the highest mountain in Japan, standing 3,776 meters tall."),
    ("en", "Sourdough bread is made by fermenting dough with wild yeast and lactobacilli."),
    ("ja", "形態素解析とは、文を単語に分割して品詞を付与する処理である。"),
    ("ja", "琵琶湖は滋賀県にある日本最大の湖である。"),
    ("ja", "天ぷらは、魚介や野菜に衣を付けて油で揚げた料理である。"),
    ("zh", "长城是中国古代的军事防御工程。"),
    ("fr", "La tour Eiffel est le monument le plus visité de Paris."),
]
doc_emb = model.encode(["passage: " + d for _, d in docs], normalize_embeddings=True)

def search(query, k=2):
    sims = doc_emb @ model.encode("query: " + query, normalize_embeddings=True)
    return [(round(float(sims[i]), 3), docs[i][0], docs[i][1][:44]) for i in np.argsort(-sims)[:k]]

for q in ["日本で一番高い山", "自己注意機構に基づくニューラルネットワーク", "パンの作り方", "How do you split Japanese text into words?",
          "largest lake in Japan", "パリの観光名所", "万里の長城"]:
    print(q)
    for hit in search(q):
        print("   ", hit)

In [ ]:
# 同じ言語の文書が上に来やすい偏りを確かめる: 同じ内容の日本語と英語の文書がある場合
pairs = ["富士山は日本で最も高い山で、標高は 3776 メートルである。", "Mount Fuji is the highest mountain in Japan, standing 3,776 meters tall."]
emb = model.encode(["passage: " + p for p in pairs], normalize_embeddings=True)
for q in ["日本で一番高い山は？", "What is the highest mountain in Japan?"]:
    sims = emb @ model.encode("query: " + q, normalize_embeddings=True)
    print(q, "→ 日本語の文書", round(float(sims[0]), 3), "/ 英語の文書", round(float(sims[1]), 3))

- 多言語の埋め込みモデルは、言語が違っても意味が近い文を近くに置くように学習されている。クエリも文書も翻訳せずに、そのまま検索できる。
- 同じ内容なら、クエリと同じ言語の文書のほうが少し高いスコアになりやすい (言語バイアス)。言語を混ぜた検索結果では、他言語の文書が下位に沈むことがある。
- 固有名詞や専門用語が言語によって大きく違う場合は弱い。クエリを機械翻訳してから検索する方法と比べて、良いほうを使う。
- BM25 のようなキーワード検索は、言語が違うと語が一致しないので使えない。ハイブリッド検索にするなら、クエリの翻訳と組み合わせる。